# Gesture Tracker

Run the cells below to start your webcam. It will:
- Track both hands using MediaPipe.
- Left hand swipes:
  - Left-to-right → shows a right arrow
  - Right-to-left → shows a left arrow
- Right hand chopping (fast downward motion) → shows a red circle

Press `x` to exit the video window.


In [1]:
import cv2
import numpy as np
import time
from collections import deque
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

# Tuning parameters
MAX_TRAIL = 6                 # how many previous positions to keep for velocity
LEFT_SWIPE_MIN_DIST = 120     # pixels horizontal displacement threshold
LEFT_SWIPE_MAX_Y_DEV = 60     # pixels vertical tolerance while swiping
LEFT_SWIPE_MIN_SPEED = 400    # pixels/second minimal speed for swipe

CHOP_MIN_SPEED = 650          # pixels/second downward speed for chop
CHOP_MIN_DIST = 100           # minimal downward displacement in pixels
CHOP_MAX_X_DEV = 80           # horizontal tolerance while chopping

SMOOTHING_ALPHA = 0.35        # EMA smoothing for landmarks

# For overlay persistence
OVERLAY_DURATION_SEC = 0.8

# Helper for exponential moving average smoothing
class EmaPoint:
    def __init__(self, alpha: float):
        self.alpha = alpha
        self.has_value = False
        self.x = 0.0
        self.y = 0.0

    def update(self, x: float, y: float):
        if not self.has_value:
            self.x, self.y = x, y
            self.has_value = True
        else:
            self.x = self.alpha * x + (1 - self.alpha) * self.x
            self.y = self.alpha * y + (1 - self.alpha) * self.y
        return self.x, self.y

# Track recent positions and timestamps for velocity computation
class MotionTracker:
    def __init__(self, maxlen: int = 6):
        self.points = deque(maxlen=maxlen)

    def add(self, x: float, y: float, t: float):
        self.points.append((x, y, t))

    def displacement(self):
        if len(self.points) < 2:
            return 0.0, 0.0, 0.0
        x0, y0, t0 = self.points[0]
        x1, y1, t1 = self.points[-1]
        dt = max(t1 - t0, 1e-6)
        return x1 - x0, y1 - y0, dt

    def velocity(self):
        dx, dy, dt = self.displacement()
        return dx / dt, dy / dt

# Gesture state to show overlays transiently
class OverlayState:
    def __init__(self):
        self.kind = None  # 'left', 'right', 'circle'
        self.until = 0.0

    def trigger(self, kind: str, duration: float = OVERLAY_DURATION_SEC):
        self.kind = kind
        self.until = time.time() + duration

    def active(self):
        return time.time() < self.until

# Drawing helpers
ARROW_COLOR = (0, 255, 255)  # yellow
CIRCLE_COLOR = (0, 0, 255)   # red

def draw_arrow(img, direction: str):
    h, w = img.shape[:2]
    center_y = h // 2
    length = int(0.25 * w)
    thickness = 10
    if direction == 'left':
        start = (w // 2 + length // 2, center_y)
        end = (w // 2 - length // 2, center_y)
    else:
        start = (w // 2 - length // 2, center_y)
        end = (w // 2 + length // 2, center_y)
    cv2.arrowedLine(img, start, end, ARROW_COLOR, thickness, tipLength=0.35)

def draw_circle(img):
    h, w = img.shape[:2]
    cv2.circle(img, (w // 2, h // 2), int(min(w, h) * 0.12), CIRCLE_COLOR, thickness=12)

# Handedness helper
def is_left(handedness):
    # MediaPipe Hands uses label 'Left' for the person's left hand
    return handedness.classification[0].label == 'Left'

# Landmark to pixel converter with smoothing
class HandAnchor:
    def __init__(self, alpha: float):
        self.smoother = EmaPoint(alpha)
        self.motion = MotionTracker(MAX_TRAIL)

    def update(self, lm, w, h, t):
        x = int(lm.x * w)
        y = int(lm.y * h)
        sx, sy = self.smoother.update(x, y)
        self.motion.add(sx, sy, t)
        return int(sx), int(sy)

# Gesture detectors
def detect_left_hand_swipe(anchor: HandAnchor, frame_w: int, frame_h: int):
    dx, dy, dt = anchor.motion.displacement()
    if dt <= 0.0:
        return None
    speed_x = abs(dx) / dt
    if abs(dy) > LEFT_SWIPE_MAX_Y_DEV:
        return None
    if speed_x < LEFT_SWIPE_MIN_SPEED:
        return None
    if abs(dx) < LEFT_SWIPE_MIN_DIST:
        return None
    return 'right' if dx > 0 else 'left'

def detect_right_hand_chop(anchor: HandAnchor):
    dx, dy, dt = anchor.motion.displacement()
    if dt <= 0.0:
        return False
    speed_y = dy / dt
    if speed_y < CHOP_MIN_SPEED:
        return False
    if dy < CHOP_MIN_DIST:
        return False
    if abs(dx) > CHOP_MAX_X_DEV:
        return False
    return True

print("Ready.")


Ready.


In [2]:
# Main loop
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)

overlay = OverlayState()

left_anchor = HandAnchor(SMOOTHING_ALPHA)
right_anchor = HandAnchor(SMOOTHING_ALPHA)

with mp_hands.Hands(
    model_complexity=0,
    max_num_hands=2,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.5,
) as hands:
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        t_now = time.time()
        results = hands.process(rgb)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_lms, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                # Use wrist (landmark 0) as the anchor point for motion
                anchor_lm = hand_lms.landmark[0]
                ax, ay = (0, 0)
                if is_left(handedness):
                    ax, ay = left_anchor.update(anchor_lm, FRAME_WIDTH, FRAME_HEIGHT, t_now)
                else:
                    ax, ay = right_anchor.update(anchor_lm, FRAME_WIDTH, FRAME_HEIGHT, t_now)

                # Draw landmarks
                mp_drawing.draw_landmarks(
                    frame,
                    hand_lms,
                    mp_hands.HAND_CONNECTIONS,
                    mp_styles.get_default_hand_landmarks_style(),
                    mp_styles.get_default_hand_connections_style(),
                )
                # Small anchor dot
                cv2.circle(frame, (ax, ay), 5, (255, 255, 255), -1)

            # Detect gestures
            swipe_dir = detect_left_hand_swipe(left_anchor, FRAME_WIDTH, FRAME_HEIGHT)
            if swipe_dir is not None:
                overlay.trigger('left' if swipe_dir == 'left' else 'right')

            if detect_right_hand_chop(right_anchor):
                overlay.trigger('circle')

        # Draw overlay if active
        if overlay.active():
            if overlay.kind == 'left':
                draw_arrow(frame, 'left')
            elif overlay.kind == 'right':
                draw_arrow(frame, 'right')
            elif overlay.kind == 'circle':
                draw_circle(frame)

        cv2.putText(frame, 'Press x to exit', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (200, 200, 200), 2)
        cv2.imshow('Gesture Tracker', frame)

        if (cv2.waitKey(1) & 0xFF) == ord('x'):
            break

cap.release()
cv2.destroyAllWindows()
print('Stopped.')


2025-08-13 09:30:47.697 python[51524:1602999] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
I0000 00:00:1755091848.142125 1602999 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1755091848.149431 1603762 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1755091848.152794 1603762 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/nathan.alam/Python-Hand-Gesture-Recognition/.venv/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() in

KeyboardInterrupt: 